# Spotify API

## Importing libraries

In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import spotipy
from spotipy.oauth2 import SpotifyOAuth
from dotenv import load_dotenv
import requests
from tqdm import tqdm

load_dotenv("../.env")

True

## Load dataframe from exploratory analysis

In [46]:
df_music = pd.read_pickle("../data/processed/df_music_features.pkl")

In [45]:
df_music["track_id"] = df_music["spotify_track_uri"].str.split(":").str[-1]

track_summary = (
    df_music
    .groupby("track_id")
    .agg(
        play_count=("track_id", "count"),
        minutes_played=("minutes_played", "sum")
    )
    .reset_index()
)

for n in [2, 3, 4, 5, 10, 20]:
    count = (track_summary["play_count"] >= n).sum()
    print(f">= {n:2d} Plays: {count:5d} Songs ({count/len(track_summary):.1%})")


>=  2 Plays: 18708 Songs (47.4%)
>=  3 Plays: 11744 Songs (29.8%)
>=  4 Plays:  8410 Songs (21.3%)
>=  5 Plays:  6452 Songs (16.3%)
>= 10 Plays:  2368 Songs (6.0%)
>= 20 Plays:   489 Songs (1.2%)


As audio features are not available with the Spotify API anymore, the Musicae API has been selected to obtain audio features. The limit for this API is 10000 requests per month. Therefore, songs with at least 4 plays (8410 songs) have been used for analysis.

In [44]:
api_key = os.getenv("MUSICAE_API_KEY")

headers = {
    "x-rapidapi-host": "spotify-extended-audio-features-api.p.rapidapi.com",
    "x-rapidapi-key": api_key
}

audio_features = []
failed_ids = []

SAVE_EVERY = 250

analysis_df = track_summary[track_summary["play_count"] >= 4].copy()

for i, track_id in enumerate(tqdm(analysis_df["track_id"], desc="Audio Features"), start=1):

    url = f"https://spotify-extended-audio-features-api.p.rapidapi.com/v1/audio-features/{track_id}"

    try:
        response = requests.get(
            url,
            headers=headers,
            timeout=15
        )

        if response.status_code == 200:
            audio_features.append(response.json())
        else:
            failed_ids.append(
                {
                    "track_id": track_id,
                    "status_code": response.status_code
                }
            )

    except Exception as e:
        failed_ids.append(
            {
                "track_id": track_id,
                "error": str(e)
            }
        )

    # Autosave
    if i % SAVE_EVERY == 0:

        pd.DataFrame(audio_features).to_csv(
            "../data/processed/audio_features_partial.csv",
            index=False
        )

        pd.DataFrame(failed_ids).to_csv(
            "../data/processed/failed_audio_features_partial.csv",
            index=False
        )

Audio Features: 100%|██████████| 8410/8410 [1:32:47<00:00,  1.51it/s]


In [51]:

audio_features_df = pd.DataFrame(audio_features)
failed_ids_df = pd.DataFrame(failed_ids)

audio_features_df.to_pickle("../data/processed/audio_features.pkl")
analysis_df.to_pickle("../data/processed/track_summary.pkl")
failed_ids_df.to_pickle("../data/processed/failed_audio_features.pkl")


print("Finished!")
print(f"Audio Features: {len(audio_features_df)}")
print(f"Failed: {len(failed_ids)}")

audio_features_df.head()

Finished!
Audio Features: 8404
Failed: 6


,acousticness,analysis_url,danceability,duration_ms,energy,id,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,time_signature,track_href,type,uri,valence
0,0.13900,https://api.spotify.com/v1/audio-analysis/004X...,0.601,123813,0.569,004X2kXNoGFFZnrWlcYLuF,0.000000,0,0.2380,-8.074,1,0.0316,139.804,4,https://api.spotify.com/v1/tracks/004X2kXNoGFF...,audio_features,spotify:track:004X2kXNoGFFZnrWlcYLuF,0.189
1,0.03900,https://api.spotify.com/v1/audio-analysis/0075...,0.451,199999,0.977,0075v1NLfKBzSBvJauGCrF,0.000978,11,0.2630,-4.182,0,0.0749,178.000,4,https://api.spotify.com/v1/tracks/0075v1NLfKBz...,audio_features,spotify:track:0075v1NLfKBzSBvJauGCrF,0.598
2,0.02480,https://api.spotify.com/v1/audio-analysis/00A4...,0.628,199245,0.956,00A4EYZsl3a1sF65zurTWr,0.836000,9,0.2510,-1.461,1,0.1340,105.987,4,https://api.spotify.com/v1/tracks/00A4EYZsl3a1...,audio_features,spotify:track:00A4EYZsl3a1sF65zurTWr,0.125
3,0.35800,https://api.spotify.com/v1/audio-analysis/00Am...,0.445,354333,0.628,00Amd2EiGo17YoZyOjk3VV,0.686000,0,0.0949,-11.268,1,0.0623,81.248,4,https://api.spotify.com/v1/tracks/00Amd2EiGo17...,audio_features,spotify:track:00Amd2EiGo17YoZyOjk3VV,0.396
4,0.00745,https://api.spotify.com/v1/audio-analysis/00B7...,0.663,251589,0.710,00B7SBwrjbycLMOgAmeIU8,0.005590,11,0.1470,-5.550,0,0.0599,120.984,4,https://api.spotify.com/v1/tracks/00B7SBwrjbyc...,audio_features,spotify:track:00B7SBwrjbycLMOgAmeIU8,0.487
